# T5.3 Gestión de secuencias con memoria contigua

En este cuaderno estudiaremos cómo representar y manipular secuencias de datos utilizando **memoria contigua** (arrays) en Python. Abordaremos las operaciones fundamentales: recorrido, búsqueda, inserción y eliminación.

> **Nota**: Este cuaderno es complementario al cuaderno [T5.2a Secuencias enlazadas](./T5.2a%20Gestión%20de%20secuencias%20enlazadas.ipynb), donde se estudian las mismas operaciones sobre secuencias con memoria dispersa (nodos enlazados). 

## 1. Representación con arrays en Python

### Convención

Representaremos un array como una **lista Python de tamaño fijo** `MAX`, donde:

- `arr`: la lista (array) de capacidad `MAX`.
- `n`: el número de elementos **efectivamente almacenados** ($0 \leq n \leq$ `MAX`).
- Las posiciones `0..n-1` contienen datos válidos.
- Las posiciones `n..MAX-1` están **sin usar** (marcadas con `None`).

> **Importante**: Deliberadamente **no usaremos** los métodos de la clase `list` de Python (`append`, `insert`, `pop`, etc.) ni la indexación negativa, ya que el objetivo es aprender a manipular arrays de forma explícita, como se haría en C/C++ o Java.

In [1]:
# Constantes y creación de un array
MAX = 8  # capacidad máxima del array

# Creamos un array vacío de tamaño MAX
arr = [None] * MAX
n = 0  # número de elementos almacenados

print(f"Array: {arr}")
print(f"Capacidad: MAX = {MAX}, elementos: n = {n}")

Array: [None, None, None, None, None, None, None, None]
Capacidad: MAX = 8, elementos: n = 0


In [2]:
# Insertamos manualmente algunos elementos
arr[0] = 8
arr[1] = 2
arr[2] = 6
arr[3] = 4
n = 4

print(f"Array: {arr}")
print(f"Elementos válidos (arr[0..{n-1}]): {arr[:n]}")

Array: [8, 2, 6, 4, None, None, None, None]
Elementos válidos (arr[0..3]): [8, 2, 6, 4]


### Función auxiliar: representación como cadena

A efectos de depuración, para visualizar el contenido del array de forma cómoda, definimos una función auxiliar que muestra todo el array, incluyendo posiciones vacías:

In [3]:
def array_to_str(arr, n):
    '''Devuelve una representación legible del array.
    
    Parámetros:
        arr: el array (lista Python).
        n (int): número de elementos válidos.
    Retorna:
        str: representación, e.g. "[8, 2, 6, 4, _, _, _, _]"
    '''
    datos = ", ".join(str(arr[i]) for i in range(n))
    vacios = ", ".join("_" for _ in range(n, len(arr)))
    if vacios:
        if n > 0:
            return f"[{datos}, {vacios}]"
        else:
            return f"[{vacios}]"
    return f"[{datos}]"

# Prueba
print(array_to_str(arr, n))

[8, 2, 6, 4, _, _, _, _]


## 2. Recorrido

El **recorrido** de un array consiste en visitar los `n` elementos válidos, realizando una cierta operación `tratar()` sobre cada dato.

El patrón de recorrido es sencillo gracias al **acceso directo** por índice. Aquí usaremos bucles `for` usando el iterador de un objeto `range`:

```python
for i in range(n):
    tratar(arr[i])
```

Comparar con el [patrón de recorrido en secuencias enlazadas](./T5.2a%20Gestión%20de%20secuencias%20enlazadas.ipynb#3.-Recorrido-de-secuencias-enlazadas):

```python
aux = arr
while aux != None:
    tratar(aux.data)
    aux = aux.next
```

En ambos casos, la complejidad temporal del recorrido es $\Theta(n)$.

### Ejemplo 1: Saturar valores a un máximo

Paralelo al [Ejemplo 4 de T5.2a](./T5.2a%20Gestión%20de%20secuencias%20enlazadas.ipynb#Ejemplo-4:-Saturar-valores-a-un-máximo).

Dado un array de enteros, reemplazar los valores mayores que `max_val` por `max_val`:

In [4]:
def saturate(arr, n, max_val):
    '''Satura los valores del array arr a un valor máximo.
    Los datos mayores que max_val se reemplazan por max_val.
    
    Parámetros:
        arr: el array.
        n (int): número de elementos válidos.
        max_val: valor máximo permitido.
    '''
    for i in range(n):
        if arr[i] > max_val:
            arr[i] = max_val

# Prueba: saturar {7, 2, 9, 4} con max_val = 3
arr = [7, 2, 9, 4, None, None, None, None]
n = 4
max_val = 3

print("Antes: ", array_to_str(arr, n))
saturate(arr, n, max_val)
print("Después:", array_to_str(arr, n))

Antes:  [7, 2, 9, 4, _, _, _, _]
Después: [3, 2, 3, 3, _, _, _, _]


## 3. Búsqueda de elementos

### Búsqueda secuencial por valor

Al igual que en las secuencias enlazadas, la búsqueda secuencial recorre el array hasta encontrar el elemento buscado o agotar los datos. Aquí también usaremos bucles `while`:

```python
i = 0
while i < n and arr[i] != d:
    i += 1
```

- Si al terminar, `i < n`: **éxito** en la búsqueda. `arr[i]` contiene el dato buscado.
- Si `i == n`: **fracaso** (el dato no se ha encontrado).

La complejidad temporal es la misma que en secuencias enlazadas: $\Omega(1) \cap O(n)$.

### Ejemplo 2: Buscar la posición de un dato

Paralelo al [Ejemplo 7 de T5.2a](./T5.2a%20Gestión%20de%20secuencias%20enlazadas.ipynb#Ejemplo-7:-Buscar-la-posición-de-un-dato).

In [5]:
def buscar(arr, n, d):
    '''Devuelve la posición de la primera ocurrencia de d en el array,
    o -1 si no se encuentra.
    
    Parámetros:
        arr: el array.
        n (int): número de elementos válidos.
        d: dato a buscar.
    Retorna:
        int: posición o -1 si no encontrado.
    '''
    i = 0
    while i < n and arr[i] != d:
        i += 1
    
    if i < n:
        return i
    else:
        return -1

# Prueba
arr = [7, 8, 9, 1, None, None, None, None]
n = 4

print(f"Array: {array_to_str(arr, n)}")
print(f"Posición de 9: {buscar(arr, n, 9)}")    # 2
print(f"Posición de 25: {buscar(arr, n, 25)}")   # -1

Array: [7, 8, 9, 1, _, _, _, _]
Posición de 9: 2
Posición de 25: -1


### Ejemplo 3: Búsqueda por posición — acceso directo $\Theta(1)$

A diferencia de las secuencias enlazadas (ver [Ejemplo 8 de T5.2a](./T5.2a%20Gestión%20de%20secuencias%20enlazadas.ipynb#Ejemplo-8:-Búsqueda-por-posición)), donde el acceso al $i$-ésimo elemento requiere un recorrido secuencial de coste $\Theta(i)$, en un array el acceso es **directo** mediante indexación:

$$\texttt{arr[i]} \quad \Rightarrow \quad \Theta(1)$$

Esta es la **principal ventaja** de la representación con memoria contigua.

In [6]:
def buscar_por_posicion(arr, n, pos):
    '''Devuelve el dato en la posición pos del array.
    
    Parámetros:
        arr: el array.
        n (int): número de elementos válidos.
        pos (int): posición del dato a obtener.
    Retorna:
        El dato almacenado en la posición pos.
    Lanza:
        IndexError si la posición no existe.
    '''
    if pos < 0 or pos >= n:
        raise IndexError(f"Posición {pos} inexistente")
    return arr[pos]  # acceso directo Θ(1)

# Prueba
arr = [7, 8, 9, 1, None, None, None, None]
n = 4

print(f"Array: {array_to_str(arr, n)}")
print(f"Dato en posición 0: {buscar_por_posicion(arr, n, 0)}")
print(f"Dato en posición 2: {buscar_por_posicion(arr, n, 2)}")

try:
    buscar_por_posicion(arr, n, 10)
except IndexError as e:
    print(f"Error: {e}")

Array: [7, 8, 9, 1, _, _, _, _]
Dato en posición 0: 7
Dato en posición 2: 9
Error: Posición 10 inexistente


## 4. Inserción de elementos

A diferencia de las secuencias enlazadas, donde la inserción solo requiere modificar enlaces ([ver Sección 5 de T5.2a](./T5.2a%20Gestión%20de%20secuencias%20enlazadas.ipynb#5.-Inserción-de-elementos)), en un array la inserción en una posición intermedia requiere **desplazar** los elementos posteriores una posición hacia la derecha para hacer hueco al nuevo dato.

Este proceso tiene un coste $\Omega(1) \cap O(n)$:

1. Verificar que hay espacio en el array (`n < MAX`).
2. Desplazar los elementos `arr[pos..n-1]` una posición a la derecha: `arr[i+1] = arr[i]`, para $i = n-1, n-2, \ldots, pos$.
3. Colocar el nuevo dato en `arr[pos]`.
4. Incrementar `n`.

<div align="center">
    <img src="fig/array_insert.png" alt="Inserción en array" style="max-width:600px"/>
</div>

### Ejemplo 4: Inserción por posición

Paralelo al [Ejemplo 10 de T5.2a](./T5.2a%20Gestión%20de%20secuencias%20enlazadas.ipynb#Ejemplo-10:-Inserción-por-posición).

> **Nota**: Dado que la inserción modifica `n`, la función **retorna** el nuevo valor de `n`.

In [7]:
def insertar(arr, n, d, pos):
    '''Inserta el dato d en la posición pos del array.
    
    Parámetros:
        arr: el array.
        n (int): número de elementos válidos.
        d: dato a insertar.
        pos (int): posición de inserción.
    Retorna:
        int: nuevo valor de n.
    Lanza:
        OverflowError si el array está lleno.
        IndexError si la posición no es válida.
    '''
    MAX = len(arr)
    if n >= MAX:
        raise OverflowError("Array lleno: no se puede insertar")
    if pos < 0 or pos > n:
        raise IndexError(f"Posición de inserción {pos} no válida")
    
    # Desplazar elementos hacia la derecha
    i = n - 1
    while i >= pos:
        arr[i + 1] = arr[i]
        i -= 1
    
    # Colocar el nuevo dato
    arr[pos] = d
    return n + 1

# Prueba: insertar en posición intermedia
arr = [8, 3, 4, None, None, None, None]
n = 3

print("Array original:         ", array_to_str(arr, n), "n = ", n)

# Insertar en posición intermedia
n = insertar(arr, n, -2, 1)
print("insertar(arr, n, -2, 1):", array_to_str(arr, n), "n = ", n)

# Insertar al principio
n = insertar(arr, n, -2, 0)
print("insertar(arr, n, -2, 0):", array_to_str(arr, n), "n = ", n)

# Insertar al final
n = insertar(arr, n, -2, n)
print(f"insertar(arr, n, -2, {n-1}):", array_to_str(arr, n), "n = ", n)

# Insertar en posición negativa
print("insertar(arr, n, -2, -1):", end=" ")
try:
    n = insertar(arr, n, -2, -1)
except IndexError as e:
    print(f"Error: {e}")

# Insertar para llenar el array
n = insertar(arr, n, -2, 4)
print("insertar(arr, n, -2, 4):", array_to_str(arr, n), "n = ", n)

# Insertar cuando el array está lleno
print("insertar(arr, n, -2, 2):", end=" ")
try:
    n = insertar(arr, n, -2, 2)
except OverflowError as e:
    print(f"Error: {e}")

Array original:          [8, 3, 4, _, _, _, _] n =  3
insertar(arr, n, -2, 1): [8, -2, 3, 4, _, _, _] n =  4
insertar(arr, n, -2, 0): [-2, 8, -2, 3, 4, _, _] n =  5
insertar(arr, n, -2, 5): [-2, 8, -2, 3, 4, -2, _] n =  6
insertar(arr, n, -2, -1): Error: Posición de inserción -1 no válida
insertar(arr, n, -2, 4): [-2, 8, -2, 3, -2, 4, -2] n =  7
insertar(arr, n, -2, 2): Error: Array lleno: no se puede insertar


### Ejemplo 5: Inserción ordenada

Paralelo al [Ejemplo 11 de T5.2a](./T5.2a%20Gestión%20de%20secuencias%20enlazadas.ipynb#Ejemplo-11:-Inserción-ordenada).

Dado un array cuyos datos están **ordenados de menor a mayor**, insertar un dato `d` manteniendo la ordenación.

La estrategia es:
1. Buscar la posición de inserción `i` (primer elemento ≥ `d`).
2. Desplazar los elementos desde `i` hacia la derecha.
3. Colocar `d` en la posición `i`.

In [12]:
def insertar_ordenado(arr, n, d):
    '''Inserta d en un array ordenado, manteniendo el orden.
    
    Parámetros:
        arr: el array ordenado.
        n (int): número de elementos válidos.
        d: dato a insertar.
    Retorna:
        int: nuevo valor de n.
    Lanza:
        OverflowError si el array está lleno.
    '''
    MAX = len(arr)
    if n >= MAX:
        raise OverflowError("Array lleno: no se puede insertar")
    
    # Buscar posición de inserción
    pos = 0
    while pos < n and d > arr[pos]:
        pos += 1
    
    # Desplazar elementos hacia la derecha
    i = n - 1
    while i >= pos:
        arr[i + 1] = arr[i]
        i -= 1
    
    # Colocar el dato
    arr[pos] = d
    return n + 1

# Prueba
arr = [1, 5, 8, 12, None, None, None]
n = 4

print("Array original:               ", array_to_str(arr, n))

n = insertar_ordenado(arr, n, 7)
print("insertar_ordenado(arr, n, 7): ", array_to_str(arr, n))

n = insertar_ordenado(arr, n, 0)
print("insertar_ordenado(arr, n, 0): ", array_to_str(arr, n))

n = insertar_ordenado(arr, n, 20)
print("insertar_ordenado(arr, n, 20):", array_to_str(arr, n))

print("insertar_ordenado(arr, n, -2):", end=' ')
try:
    n = insertar_ordenado(arr, n, -2)
except OverflowError as e:
    print(f"Error: {e}")



Array original:                [1, 5, 8, 12, _, _, _]
insertar_ordenado(arr, n, 7):  [1, 5, 7, 8, 12, _, _]
insertar_ordenado(arr, n, 0):  [0, 1, 5, 7, 8, 12, _]
insertar_ordenado(arr, n, 20): [0, 1, 5, 7, 8, 12, 20]
insertar_ordenado(arr, n, -2): Error: Array lleno: no se puede insertar


### 4.1 Inserción con redimensionamiento automático del array

En las implementaciones anteriores ([Ejemplo 4](#Ejemplo-4:-Inserción-por-posición) y [Ejemplo 5](#Ejemplo-5:-Inserción-ordenada)), asumíamos que si el array estaba lleno (`n >= MAX`), la inserción fallaba lanzando una excepción `OverflowError`. 

Sin embargo, en la práctica (como hacen internamente las listas dinámicas de Python o la clase `std::vector` de C++), cuando el array se llena y queremos insertar un nuevo elemento, se realiza un **redimensionado automático** de la estructura.

Este (costoso) proceso consta de los siguientes pasos:
1. **Crear un nuevo array** con una capacidad significativamente mayor (típicamente el **doble de la capacidad original**, para amortizar el coste de la copia).
2. **Copiar todos los elementos** del array antiguo al nuevo array.
3. **Reemplazar la referencia** al array antiguo por el nuevo array de mayor capacidad.
4. **Proceder con la inserción estándar** (desplazamiento de elementos a partir de la posición `pos` e inserción del nuevo dato).


> IMPORTANTE: dado que en Python los argumentos a funciones se pasan por valor, si reasignamos la variable local `arr` a una nueva lista con mayor capacidad dentro de la función, el llamador no verá el cambio en su variable original. Por ello, la función que redimensiona debe **retornar obligatoriamente la nueva secuencia y la nueva talla**, de forma análoga a cómo retornamos la nueva cabeza en las secuencias enlazadas.

In [ ]:
def insertar_redimensionando(arr, n, d, pos):
    '''Inserta el dato d en la posición pos del array, redimensionándolo si está lleno.
    
    Si el array está lleno (n >= len(arr)), se crea un nuevo array con el doble
    de la capacidad original, se copian los elementos existentes y se realiza la inserción.
    
    Parámetros:
        arr: el array (lista Python).
        n (int): número de elementos válidos.
        d: dato a insertar.
        pos (int): posición de inserción.
    Retorna:
        (nuevo_arr, nuevo_n): tupla con el nuevo array y la nueva talla.
    Lanza:
        IndexError si la posición no es válida.
    '''
    MAX = len(arr)
    if pos < 0 or pos > n:
        raise IndexError(f"Posición de inserción {pos} no válida")
    
    # Redimensionado en caso de estar lleno
    if n >= MAX:
        nueva_capacidad = MAX * 2
        print(f"- [ADVERTENCIA] ¡Array lleno! Redimensionando de capacidad {MAX} a {nueva_capacidad}...")

        # 1. Duplicar la capacidad original
        nuevo_arr = [None] * nueva_capacidad
        
        # 2. Copiar los elementos del array antiguo al nuevo
        for i in range(n):
            nuevo_arr[i] = arr[i]
            
        # 3. Actualizar referencia al array
        arr = nuevo_arr
        
    # 4. Realizar la inserción estándar con desplazamiento
    i = n - 1
    while i >= pos:
        arr[i + 1] = arr[i]
        i -= 1
        
    arr[pos] = d

    return arr, n + 1

# Prueba de inserción con redimensionado automático
arr = [8, 3, 4, None]  # MAX = 4
n = 3

print("Array original:                           ", array_to_str(arr, n), f"n = {n}, capacidad = {len(arr)}")

# Insertamos -> hay espacio, no provoca redimensionado
arr, n = insertar_redimensionando(arr, n, -2, 1)
print("insertar_redimensionando(arr, n, -2, 1):  ", array_to_str(arr, n), f"n = {n}, capacidad = {len(arr)}")

# Insertamos en un array lleno -> provoca redimensionado automático
arr, n = insertar_redimensionando(arr, n, -2, 3)
print("insertar_redimensionando(arr, n, -2, 3):  ", array_to_str(arr, n), f"n = {n}, capacidad = {len(arr)}")

# Insertamos de nuevo -> hay espacio, no provoca redimensionado
arr, n = insertar_redimensionando(arr, n, -99, 0)
print("insertar_redimensionando(arr, n, -99, 0): ", array_to_str(arr, n), f"n = {n}, capacidad = {len(arr)}")

Array original:                            [8, 3, 4, _] n = 3, capacidad = 4
insertar_redimensionando(arr, n, -2, 1):   [8, -2, 3, 4] n = 4, capacidad = 4
- [ADVERTENCIA] ¡Array lleno! Redimensionando de capacidad 4 a 8...
insertar_redimensionando(arr, n, -2, 3):   [8, -2, 3, -2, 4, _, _, _] n = 5, capacidad = 8
insertar_redimensionando(arr, n, -99, 0):  [-99, 8, -2, 3, -2, 4, _, _] n = 6, capacidad = 8


## 5. Eliminación de elementos

De forma análoga a la inserción, la eliminación en un array requiere **desplazar** los elementos posteriores una posición hacia la izquierda para mantener la contigüidad. Esto se denomina el proceso de **compactación** del array y tiene un coste $\Omega(1) \cap O(n)$.

Comparar con la eliminación en secuencias enlazadas ([ver Sección 6 de T5.2a](./T5.2a%20Gestión%20de%20secuencias%20enlazadas.ipynb#6.-Eliminación-de-elementos)), donde solo se modifican enlaces (*bypass*).

El proceso de eliminación en la posición `pos` es:
1. Desplazar los elementos `arr[pos+1..n-1]` una posición a la izquierda: `arr[i-1] = arr[i]`, para $i = pos+1, pos+2, \ldots, n-1$.
2. Marcar la última posición ocupada como vacía: `arr[n-1] = None`.
3. Decrementar `n`.

<div align="center">
    <img src="fig/array_delete.png" alt="Eliminación en array" style="max-width:600px"/>
</div>

### Ejemplo 6: Eliminar por posición

Paralelo al [Ejemplo 12 de T5.2a](./T5.2a%20Gestión%20de%20secuencias%20enlazadas.ipynb#Ejemplo-12:-Eliminar-por-posición).

In [10]:
def eliminar_pos(arr, n, pos):
    '''Elimina el elemento en la posición pos del array.
    
    Parámetros:
        arr: el array.
        n (int): número de elementos válidos.
        pos (int): posición del elemento a eliminar.
    Retorna:
        int: nuevo valor de n.
    Lanza:
        IndexError si la posición no es válida o si la secuencia está vacía.
    '''
    if n == 0:
        raise IndexError("Secuencia vacía")
    if pos < 0 or pos >= n:
        raise IndexError(f"Posición {pos} no válida")
    
    # Desplazar elementos hacia la izquierda
    i = pos + 1
    while i < n:
        arr[i-1] = arr[i]
        i += 1
    
    # Limpiar la última posición (= eliminar referencia al objeto)
    arr[n - 1] = None
    return n - 1

# Prueba
arr = [8, 2, 6, None, None]
n = 3

print("Array original:     ", array_to_str(arr, n))

n = eliminar_pos(arr, n, 2)
print("Eliminar pos. 2:    ", array_to_str(arr, n))

n = eliminar_pos(arr, n, 0)
print("Eliminar pos. 0:    ", array_to_str(arr, n))

# Eliminar en posición negativa
print("Eliminar pos. -1:   ", end=" ")
try:
    n = eliminar_pos(arr, n, -1)
except IndexError as e:
    print(f"Error: {e}")

# Eliminar en posición inválida
print("Eliminar pos. 5:    ", end=" ")
try:
    n = eliminar_pos(arr, n, 5)
except IndexError as e:
    print(f"Error: {e}")

# Eliminar el último elemento
n = eliminar_pos(arr, n, 0)
print("Eliminar pos. 0:    ", array_to_str(arr, n))

# Eliminar en secuencia vacía
print("Eliminar en vacía:  ", end=" ")
try:
    n = eliminar_pos(arr, n, 0)
except IndexError as e:
    print(f"Error: {e}")

Array original:      [8, 2, 6, _, _]
Eliminar pos. 2:     [8, 2, _, _, _]
Eliminar pos. 0:     [2, _, _, _, _]
Eliminar pos. -1:    Error: Posición -1 no válida
Eliminar pos. 5:     Error: Posición 5 no válida
Eliminar pos. 0:     [_, _, _, _, _]
Eliminar en vacía:   Error: Secuencia vacía


### Ejemplo 7: Eliminar la primera ocurrencia de un dato

Paralelo al [Ejemplo 13 de T5.2a](./T5.2a%20Gestión%20de%20secuencias%20enlazadas.ipynb#Ejemplo-13:-Eliminar-la-primera-ocurrencia-de-un-dato).

La estrategia consiste en:
1. Buscar la posición del dato.
2. Si se encuentra, eliminar en esa posición (desplazamiento hacia la izquierda).

In [ ]:
def eliminar_dato(arr, n, d):
    '''Elimina, si existe, la primera ocurrencia del dato d.
    
    Parámetros:
        arr: el array.
        n (int): número de elementos válidos.
        d: dato a eliminar.
    Retorna:
        int: nuevo valor de n.
    '''
    # Buscar d, aprovechamos la función de búsqueda ya implementada :)
    pos = buscar(arr, n, d)
    
    if pos != -1:  # éxito en la búsqueda; aprovechamos la función de eliminación por posición ya implementada :)
        n = eliminar_pos(arr, n, pos) # 
    
    return n

# Prueba
arr = [7, 1, 9, 4, None, None, None, None]
n = 4

print("Original:            ", array_to_str(arr, n))

n = eliminar_dato(arr, n, 9)
print("Eliminar 9:          ", array_to_str(arr, n))

n = eliminar_dato(arr, n, 7)
print("Eliminar 7:          ", array_to_str(arr, n))

n = eliminar_dato(arr, n, 25)
print("Eliminar 25 (fallo): ", array_to_str(arr, n))

Original:             [7, 1, 9, 4, _, _, _, _]
Eliminar 9:           [7, 1, 4, _, _, _, _, _]
Eliminar 7:           [1, 4, _, _, _, _, _, _]
Eliminar 25 (fallo):  [1, 4, _, _, _, _, _, _]


## 6. Comparativa de costes temporales con secuencias enlazadas

| Operación | Memoria contigua (array) | Memoria dispersa (nodos enlazados) |
|---|---|---|
| **Recorrido** | $\Theta(n)$ | $\Theta(n)$ |
| **Búsqueda por valor** | $\Omega(1) \cap O(n)$ | $\Omega(1) \cap O(n)$ |
| **Búsqueda por posición** | $\Theta(1)$ — acceso directo | $\Omega(1) \cap O(n)$ — secuencial |
| **Inserción** | $\Omega(1) \cap O(n)$ — desplazamiento | $\Omega(1) \cap O(n)$ — localización + enlace |
| **Eliminación** | $\Omega(1) \cap O(n)$ — compactación | $\Omega(1) \cap O(n)$ — localización + bypass |

> **Observación clave**: La principal ventaja de los arrays es el **acceso directo** por posición en $\Theta(1)$. Su principal desventaja es el **desplazamiento de elementos** en memoria requerido por las operaciones de inserción y eliminación, y por la limitación de espacio de almacenamiento. Las secuencias enlazadas presentan la situación inversa: inserción/eliminación eficientes (una vez localizado el punto), pero acceso por posición costoso.

En los próximos cuadernos implementaremos las tres EDLs principales (Pila, Cola, Lista) utilizando **ambas representaciones**, lo que permitirá apreciar cómo cada operación se beneficia de una u otra aproximación.